# PCA + ACF curve analysis


## Set up
### Imports

In [ ]:
# pathing
from pathlib import Path
import sys
import yaml
import os

# Locate repo root
repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "config" / "config.yml").exists()
)

# Add code directory to import path
code_dir = repo_root / "code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

# Load config
config_path = repo_root / "config" / "config.yml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

In [ ]:
# Pacakage imports
from IPython.display import display
from IPython import get_ipython
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
import numpy as np
from sklearn.decomposition import PCA
from utils.preprocessing.io import load_stage_npz, derive_session_id_from_path
from utils.eda import run_pca_eda
from scipy.signal import welch

In [ ]:
# Inline backend
USE_WIDGET = False
ipy = get_ipython()
backend_mode = "inline"
if ipy is not None:
    try:
        ipy.run_line_magic("matplotlib", "inline")
    except Exception:
        pass
print(f"Matplotlib backend: {matplotlib.get_backend()} | mode={backend_mode}")

# Core paths
deriv_root = repo_root / config["paths"]["preprocessing"]
subjects = config["subjects"]["all"]
sample_subject = config["subjects"]["default"]
data_mode = "zscore"
sample_session_idx = 2
sample_frame_idx = 0

# Analysis params
# add main pca params to config and load from there

### Load data

---
## PCA 
This section will compute the pca and plot the scree+cumulative EVR plots, the pc score time traces, and the spatial loading maps

### Section Outputs
store outputs in variables for later use: pca, scores, components, explained_var_ratio, mean_, spatial_maps

---
## Effect on ACF curve
This section will look at the effect of dropping pcs on the acf curve. It will plot ACF curves vs lag for: raw (or full recon) + each drop condition (k_drop grid), ΔACF vs lag (ACF_drop − ACF_raw/full) for each drop condition (much easier to interpret than stacked lines), Summary metrics vs k_drop (AUC of ACF over lag window; half-life)

### Section Outputs
store outputs in variables for later use: acf_by_condition, delta_acf_by_condition, summary_df

---
## PCA Robustness checks
this section will check the pca reconstruction. it will plot raw vs recon overlay (and residual) for a representative voxel/ROI/time window, uncertainty on ACF / ΔACF (bootstrap across windows/sessions/voxels/ROIs; show CI band), Frequency-domain check (PSD) pre/post drop (optional but often helpful). It will also compare the reconstructions of the different conditons (same plot just different colour traces per condition.)

### Section Outputs
store outputs in variables for later use: recons_by_condition, residuals_by_condition, psd_by_condition

---
## Video comparison
this section will make us of hf.make_conjoined_video to compare the frames before and after k_drop = 1